<!--
SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: Apache-2.0
-->

# Aim
The purpose of this notebook is to demo AITune module inspecting and automated wrapping based on .

To run this, install extras `demo`.

In [ ]:
%load_ext autoreload
%autoreload 2
%cd ..
%pwd
%pip install diffusers<=0.34.0 transformers>=4.54,<5

In [ ]:
import aitune

# Inspecting Stable Diffusion pipeline

Let's use Stable Diffusion 3 as example pipeline for inspect and automated wrapping.

In [ ]:
import diffusers

model_name = "stabilityai/stable-diffusion-3-medium-diffusers"
pipe = diffusers.DiffusionPipeline.from_pretrained(model_name)

pipe.to("cuda")

Inspect the pipeline on sample input prompt:

In [ ]:
input_data = [{"prompt": "A beautiful landscape with mountains and a lake"}]

modules_info = aitune.torch.inspect(pipe, input_data)

Once the pipeline has been evaluated we got collected information about the top level executed modules withing the pipeline. First, let's see the detail information about the modules:

In [ ]:
modules_info.describe()

Example output:

In [ ]:
Module Execution Summary:
==========================================================================================================================================
    Module Name        Calls    Total Time (s)    Avg Time (s)    % of Total    # of params      # of layers           precisions        
------------------------------------------------------------------------------------------------------------------------------------------
text_encoder                 2           0.3582           0.1791        2.12%        340387840                1              torch.float32
unet                        50          16.1990           0.3240       95.72%        865910724                9              torch.float32
vae.decoder                  1           0.2125           0.2125        1.26%         49490179                6              torch.float32
vae.post_quant_conv          1           0.0016           0.0016        0.01%               20                0              torch.float32
------------------------------------------------------------------------------------------------------------------------------------------
Total execution time: 16.922543 seconds
==========================================================================================================================================

The table present information about each module with details:
- how many times each module has been executed
- what was the total inference time that was spend during inference call in each module
- what was the average time of single module forward call
- the percentage of total time module execution time of total pipeline execution time

And some more information about number of parameters, layers and precision of each module.

# Using inspect information for tuning

Once the information about modules was collected we can move to wrapping the modules for further tuning. Here we have couple of options that could be applied for selecting the modules for tuning.

### Selecting all modules

First and simplest is selecting all modules for tuning with following code:

In [ ]:
modules = modules_info.get_modules()

### Selecting modules conditionally

In certain situation we would like to wrap only the top executed modules based on some conditions like:
- top k modules based on % of total execution time
- minimal % of total execution time

For such purpose use different method:

In [ ]:
# Top 1 module
modules = modules_info.get_modules(limit=1)

# Minimum 40% of total execution time
modules = modules_info.get_modules(min_execution_percentage=0.4)

### Wrapping and tunning modules

Once we have selected modules we can automatically wrap them and perform tuning:

In [ ]:
from aitune.torch.backend import TensorRTBackend, TorchInductorBackend
from aitune.torch.tune_strategy import FirstWinsStrategy

# Wrap modules
pipe = aitune.torch.wrap(pipe, modules, strategy=FirstWinsStrategy(backends=[TensorRTBackend(), TorchInductorBackend()]))

Finally, execute tune:

In [ ]:
# Tune modules
pipe = aitune.torch.tune(pipe, input_data, batch_sizes=[1])